# unbox-args-tensor-to-array — ex3: unbox_kwargs: dict-comprehension unboxes MiniTensor values, preserves keys + order

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `unbox-args-tensor-to-array`. Running the final beacon cell reports progress against the `Backprop: Unbox Tensor args to array` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Unbox Tensor args to array` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`unbox-args-tensor-to-array`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "unbox-args-tensor-to-array"
DD_SUBTOPIC = "Backprop: Unbox Tensor args to array"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `unbox_kwargs` — unboxing the dict-shaped container

Ex1 unboxed the positional args tuple. Ex2 recursed into nested list/tuple containers. The third facet completes the picture: kwargs are a `dict[str, value]` — a different container that also needs unboxing because nothing prevents a user from passing a MiniTensor as a kwarg.

```python
def unbox_kwargs(kwargs: dict) -> dict:
    return {
        k: v.array if isinstance(v, MiniTensor) else v
        for k, v in kwargs.items()
    }
```

**Why this is non-trivial despite the one-liner.** Many ops take tensors as kwargs: `t.where(cond, x, y)` (cond as a kwarg in custom wrappers), `t.scatter(input, dim, index, src)` (`src` as kwarg). Without kwarg unboxing, the raw torch fn receives a MiniTensor and crashes with `AttributeError: 'MiniTensor' has no attribute ...`.

**Why a dict-comprehension preserves key order.** Python 3.7+ guarantees dict insertion order. Iterating `kwargs.items()` and rebuilding via comprehension preserves it. Important when the raw fn relies on kwarg ORDER for its repr or for downstream caches.

**Dual of ex1.** Same `isinstance(_, MiniTensor)` test, same `.array` swap. The only difference is the container: tuple vs dict. Together with the positional unbox + the nested-container unbox from ex2, this covers every shape user code can throw at the wrapper.

### Exercise 3 — unbox_kwargs: dict-comprehension unboxes MiniTensor values, preserves keys + order

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the unboxing rule across a kwargs dict: replace each MiniTensor-valued entry with its .array, leaving keys and insertion order intact and all non-MiniTensor values pass-through.
> Keywords: unbox, kwargs, dict, wrap-forward, container
> ```

**KCs targeted:** `unbox-kwargs-dict-comprehension`, `preserve-keys-and-insertion-order`

Implement `ex3_unbox_kwargs(kwargs)`. Same unbox rule as ex1 (positional args) and ex2 (nested lists/tuples), but for a DICT container.

Behavior:

- For each `(k, v)` in `kwargs.items()`:
  - If `isinstance(v, MiniTensor)`: replace `v` with `v.array`.
  - Otherwise: keep `v` unchanged (identity pass-through).
- Keys are preserved exactly.
- Insertion order is preserved (Python 3.7+ dict semantics).
- Return type is `dict` (NOT `collections.OrderedDict`, NOT a generator).
- Original `kwargs` dict is NOT mutated — return a NEW dict.

Examples:

```
unbox_kwargs({'src': m1, 'dim': 1})    → {'src': m1.array, 'dim': 1}
unbox_kwargs({'min': 0.0, 'max': 1.0}) → {'min': 0.0, 'max': 1.0}   # pass-through
unbox_kwargs({})                        → {}
```

Constraints:
- Use `isinstance(v, MiniTensor)` — NOT duck-typing on `.array` (would catch numpy ndarrays which have an `.array` protocol).
- Raw `torch.Tensor` values MUST pass through (not MiniTensor → unchanged).
- The unboxed `.array` value MUST be identity-equal to the original (`result[k] is v.array`).

In [ ]:
def ex3_unbox_kwargs(kwargs: dict) -> dict:
    """Replace MiniTensor values with .array; pass-through everything else."""
    raise NotImplementedError()


def _test_ex3():
    def _test_ex3():
        # === Empty dict ===
        assert ex3_unbox_kwargs({}) == {}

        # === All non-MiniTensor values pass through ===
        assert ex3_unbox_kwargs({'dim': 1, 'keepdim': True}) == {'dim': 1, 'keepdim': True}
        assert ex3_unbox_kwargs({'min': 0.0, 'max': 1.0}) == {'min': 0.0, 'max': 1.0}

        # === Single MiniTensor unwrapped ===
        raw = t.tensor([1.0, 2.0, 3.0])
        m = MiniTensor(raw)
        result = ex3_unbox_kwargs({'src': m})
        assert isinstance(result, dict)
        assert set(result.keys()) == {'src'}
        assert result['src'] is raw, (
            'unboxed value must BE the same torch.Tensor object (identity)'
        )

        # === Mixed: MiniTensor + Python scalars ===
        result = ex3_unbox_kwargs({'src': m, 'dim': 1, 'keepdim': True})
        assert result['src'] is raw
        assert result['dim'] == 1
        assert result['keepdim'] is True

        # === Raw torch.Tensor values pass through (NOT MiniTensor → not unboxed) ===
        raw_passthrough = t.tensor([9.0])
        result = ex3_unbox_kwargs({'tensor_arg': raw_passthrough})
        assert result['tensor_arg'] is raw_passthrough, (
            'raw torch.Tensor must pass through (only MiniTensor unboxes)'
        )

        # === Numpy ndarray pass-through (don't be fooled by .array protocol) ===
        arr = np.array([1.0, 2.0, 3.0])
        result = ex3_unbox_kwargs({'arr': arr})
        assert result['arr'] is arr, (
            'np.ndarray must pass through — isinstance(MiniTensor) is False '
            '(do not duck-type on .array)'
        )

        # === Insertion order preserved ===
        inp = {}
        inp['z'] = m
        inp['a'] = 1.0
        inp['m'] = True
        result = ex3_unbox_kwargs(inp)
        assert list(result.keys()) == ['z', 'a', 'm'], (
            f'insertion order must be preserved, got {list(result.keys())}'
        )

        # === Original kwargs NOT mutated ===
        inp = {'src': m, 'dim': 1}
        result = ex3_unbox_kwargs(inp)
        assert inp == {'src': m, 'dim': 1}, (
            f'input dict must not be mutated; got {inp}'
        )
        assert inp['src'] is m, 'value identity preserved in input'

        # === Several MiniTensors in one dict ===
        m2 = MiniTensor(t.tensor([4.0]))
        m3 = MiniTensor(t.tensor([5.0]))
        result = ex3_unbox_kwargs({'a': m, 'b': m2, 'c': m3})
        assert result['a'] is m.array
        assert result['b'] is m2.array
        assert result['c'] is m3.array

        # === Other non-MiniTensor types pass through (None, str, tuple, list) ===
        result = ex3_unbox_kwargs({'opt': None, 'name': 'x', 'shape': (3, 4), 'lst': [1, 2]})
        assert result == {'opt': None, 'name': 'x', 'shape': (3, 4), 'lst': [1, 2]}

        # === Return is dict (not generator, not OrderedDict-required, not list of tuples) ===
        result = ex3_unbox_kwargs({'x': 1})
        assert type(result) is dict, (
            f'must return a plain dict, got {type(result).__name__}'
        )

        # === The use case end-to-end: kwargs-as-raw-tensor flows into a torch op ===
        # t.add accepts an 'alpha' scalar kwarg — but for a MiniTensor src kwarg case,
        # something like t.scatter takes 'src' as a kwarg-positional.
        x = t.tensor([0.0, 0.0, 0.0])
        m_src = MiniTensor(t.tensor([1.0, 2.0, 3.0]))
        kw = ex3_unbox_kwargs({'src': m_src})
        # Simulate the raw op consuming kw — just identity here.
        assert kw['src'] is m_src.array
        print('ex3 ✓')

    _test_ex3()
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_unbox_kwargs(kwargs):
    return {
        k: v.array if isinstance(v, MiniTensor) else v
        for k, v in kwargs.items()
    }
```

**Dict comprehension is the canonical one-liner.** Same shape as ex1's tuple comprehension — `isinstance(v, MiniTensor)` gate, `.array` swap on hit, identity pass-through on miss. The difference is just the container: dict vs tuple.

**Why a new dict, not in-place mutation.** Callers retain the original `kwargs` dict for `Recipe.kwargs` storage — the Recipe wants the ORIGINAL types (MiniTensor) for graph-traversal purposes, while the raw forward fn needs the unboxed versions. Mutating in place would corrupt the Recipe.

**`isinstance` over duck-typing.** Numpy ndarrays expose an `.array` interface protocol — duck-typing on `.array` would incorrectly unbox them. `isinstance(v, MiniTensor)` is precise: only our wrapper class gets the unboxing treatment.

**Why this is the third facet of the atom.** Ex1 covered the tuple container (positional args). Ex2 covered nested list/tuple containers (cat / stack signatures). Ex3 covers the dict container (kwargs). Together they handle every Python container shape the wrapper layer encounters — and the rule is the same scalar check applied to each shape's natural traversal.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()